In [1]:
# Cell 1: Environment Setup

include("scripts/buildaux_helpers.jl")
include("scripts/buildaux_dictionaries.jl")
using .BuildAuxHelpers
using .BuildAuxDictionaries
using OMJulia

# --- Configuration ---

# 1. Define the directory containing your models
MODEL_DIR = abspath("MyNordic")

# 2. Path to your local package
MODELS_PKG_PATH = joinpath(MODEL_DIR, "package.mo")

# 3. Define the root model name 
MODEL = "MyNordic.TestCase"

# 4. Path to the Dynawo package.mo
DYNAWO_PKG_PATH = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"

# 5. Path to the Modelica package.mo
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"


"/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"

In [2]:
# Cell 2: OpenModelica Setup + User's Case Validation

# 1. Start OMC and load libraries
omc = OMJulia.OMCSession()
om_send(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
om_send(omc, "loadModel(Complex)")
om_send(omc, "loadModel(ModelicaServices)")
om_send(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")

# 2. Load the user's case and validate it
om_send(omc, "loadFile(\"$MODELS_PKG_PATH\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($MODEL)", parsed=false)
println(chk)

[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.wMwlQYSssA"


OMC -> loadFile("/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")
OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")
OMC -> loadFile("/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/MyNordic/package.mo")
OMC -> clearMessages()
OMC -> checkModel(MyNordic.TestCase)
"Check of MyNordic.TestCase completed successfully.
Class MyNordic.TestCase has 6622 equation(s) and 6622 variable(s).
2846 of these are trivial equation(s)."



In [3]:
# Cell 3: Get Inheritance + Auxiliary Model Setup

# 1. Build the inheritance chain
chain = get_inheritance_chain(omc, MODEL)
println(chain)

# 2. Auxiliary package naming + paths
AUX_PACKAGE = split(MODEL, ".")[1] * "_auxiliary"
AUX_DIR = joinpath(dirname(MODEL_DIR), AUX_PACKAGE)
AUX_ROOT_MODEL = AUX_PACKAGE * "." * split(MODEL, ".")[end] * "_auxiliary"

AUX_NAME_MAP = Dict(model => AUX_PACKAGE * "." * split(model, ".")[end] * "_auxiliary" for model in chain)
AUX_PACKAGE_FILE = joinpath(AUX_DIR, "package.mo")
AUX_ORDER_FILE = joinpath(AUX_DIR, "package.order")

OMC -> getInheritedClasses(MyNordic.TestCase)
OMC -> getInheritedClasses(MyNordic.FullDynamicModel)
OMC -> getInheritedClasses(MyNordic.NetworkWithAlphaBetaLoads)
OMC -> getInheritedClasses(MyNordic.Network)
["MyNordic.Network", "MyNordic.NetworkWithAlphaBetaLoads", "MyNordic.FullDynamicModel", "MyNordic.TestCase"]


"/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/MyNordic_auxiliary/package.order"

In [4]:
# Cell 4: Component-Specific INIT Model Selection + Slack Component Selection

INIT_MODEL_BY_COMPONENT = Dict{String, String}(
    "g01" => "GeneratorSynchronousExt3W_INIT",
    "g02" => "GeneratorSynchronousExt3W_INIT",
    "g03" => "GeneratorSynchronousExt3W_INIT",
    "g04" => "GeneratorSynchronousExt3W_INIT",
    "g05" => "GeneratorSynchronousExt3W_INIT",
    "g06" => "GeneratorSynchronousExt4W_INIT",
    "g07" => "GeneratorSynchronousExt4W_INIT",
    "g08" => "GeneratorSynchronousExt3W_INIT",
    "g09" => "GeneratorSynchronousExt3W_INIT",
    "g10" => "GeneratorSynchronousExt3W_INIT",
    "g11" => "GeneratorSynchronousExt3W_INIT",
    "g12" => "GeneratorSynchronousExt3W_INIT",
    "g13" => "GeneratorSynchronousExt3W_INIT",
    "g14" => "GeneratorSynchronousExt4W_INIT",
    "g15" => "GeneratorSynchronousExt4W_INIT",
    "g16" => "GeneratorSynchronousExt4W_INIT",
    "g17" => "GeneratorSynchronousExt4W_INIT",
    "g18" => "GeneratorSynchronousExt4W_INIT",
    "g19" => "GeneratorSynchronousExt3W_INIT",
    "g20" => "GeneratorSynchronousExt3W_INIT",
)

SLACK_COMPONENT = "g20"

"g20"

In [ ]:
# Cell 5: Main Auxiliary Build Pipeline

# Main script

# Create an empty auxiliary package in OpenModelica
om_send(omc, "deleteClass($AUX_PACKAGE)")
om_send(omc, "clearMessages()")
om_send(omc, "loadString(\"within ; package $AUX_PACKAGE end $AUX_PACKAGE;\")")

# Build local component dictionaries and cumulative patch contexts across the inheritance chain
components_by_model = Dict{String, Dict{String, Dict{String, Any}}}()
patch_components_by_model = Dict{String, Dict{String, Dict{String, Any}}}()
cumulative_components = Dict{String, Dict{String, Any}}()
global_blacklist_names = Set{String}()
for model in chain
    model_components = get_all_components(omc, model)
    components_by_model[model] = model_components
    union!(global_blacklist_names, collect_cleanup_component_names(model_components))

    merge!(cumulative_components, model_components)
    patch_components_by_model[model] = copy(cumulative_components)
end

# Loop over the inheritance chain and transform each class
for model in chain
    aux_model = AUX_NAME_MAP[model]
    aux_name = split(aux_model, ".")[end]

    println("Transforming $model -> $aux_model")
    # Copy original class into the auxiliary package
    om_send(omc, "copyClass($model, \"$aux_name\", $AUX_PACKAGE)")

    # Use the local component dictionary from the original class
    components = components_by_model[model]

    # Apply dictionary-driven replacements
    apply_replacements!(omc, model, aux_model, REPLACEMENTS, components, SLACK_COMPONENT)
    # Delete connections to the sources
    delete_connections!(omc, aux_model, components; global_targets = global_blacklist_names)

    # Delete sources
    delete_components!(omc, aux_model, components)

    # Add INIT models (dictionary-driven)
    add_init_models!(omc, model, aux_model, INIT_MODELS, INIT_MODEL_BY_COMPONENT, components, SLACK_COMPONENT)

    # Add extra modifiers necessary for the load-flow
    apply_LF_modifiers!(omc, model, aux_model, INIT_MODELS, components)

    # Add Initial equations
    add_init_equations!(omc, model, aux_model, components, INIT_MODELS, INIT_MODEL_BY_COMPONENT, SLACK_COMPONENT)
end

# Create the auxiliary package folder
mkpath(AUX_DIR)

# Write package.mo
open(AUX_PACKAGE_FILE, "w") do io
    print(io, "within ;\npackage $AUX_PACKAGE\nend $AUX_PACKAGE;\n")
end

# Write package.order
open(AUX_ORDER_FILE, "w") do io
    for model in chain
        println(io, split(AUX_NAME_MAP[model], ".")[end])
    end
end

# Save each transformed auxiliary class
for model in chain
    aux_model = AUX_NAME_MAP[model]
    aux_name = split(aux_model, ".")[end]
    aux_file = joinpath(AUX_DIR, aux_name * ".mo")

    # Apply equation cleanup with components visible through this class' inheritance chain
    patch_components = patch_components_by_model[model]
    clean_aux_equations!(omc, aux_model, SLACK_COMPONENT; components = patch_components)

    # Get the current transformed class text from OpenModelica
    txt = String(om_send(omc, "listFile($aux_model)"))

    # Rewrite extends(...) so the auxiliary classes inherit from the auxiliary parents
    txt = rewrite_aux_extends(txt, AUX_NAME_MAP)

    # Save the auxiliary class
    open(aux_file, "w") do io
        print(io, txt)
        endswith(txt, "\n") || print(io, "\n")
    end
end

# Re-load generated auxiliary package from disk and validate
om_send(omc, "deleteClass($AUX_PACKAGE)")
om_send(omc, "loadFile(\"$AUX_PACKAGE_FILE\")")
om_send(omc, "clearMessages()")
chk = om_send(omc, "checkModel($AUX_ROOT_MODEL)", parsed=false)
println(chk)


### Optional diagnostics
Run the next cell only if the main build/check cell fails or you need detailed OpenModelica messages.


In [6]:
# Cell 7: OMC diagnostics for failed checks

# Run this cell after the build/check cell to isolate OpenModelica failures.

function _print_omc_errors(label::String)
    raw = String(sendExpression(omc, "getErrorString()", parsed=false))
    txt = strip(replace(raw, "\"" => ""))
    println("\n[$label] getErrorString()")
    if isempty(txt)
        println("<no messages>")
    else
        println(raw)
    end
end

function _check_and_report(model_name::String)
    sendExpression(omc, "clearMessages()")
    println("\n=== checkModel($model_name) ===")
    chk = sendExpression(omc, "checkModel($model_name)", parsed=false)
    println(chk)
    _print_omc_errors(model_name)
    return chk
end

println("=== OMC diagnostics start ===")
_print_omc_errors("after previous cell")

# 1) Root auxiliary model
_check_and_report(AUX_ROOT_MODEL)

# 2) Every transformed class in the chain
for model in chain
    aux_model = AUX_NAME_MAP[model]
    _check_and_report(aux_model)
end

# 3) Optional compile-time expansion (often gives clearer errors)
sendExpression(omc, "clearMessages()")
println("\n=== instantiateModel($AUX_ROOT_MODEL) ===")
inst = sendExpression(omc, "instantiateModel($AUX_ROOT_MODEL)", parsed=false)
inst_s = String(inst)
if startswith(strip(inst_s), "Error")
    println(inst_s)
end
_print_omc_errors("instantiateModel")

println("=== OMC diagnostics end ===")


=== OMC diagnostics start ===

[after previous cell] getErrorString()
"Notification: Modelica requested package Complex of version 3.2.3. Complex 4.1.0 is used instead which states that it is fully compatible without conversion script needed.
Notification: Modelica requested package ModelicaServices of version 3.2.3. ModelicaServices 4.1.0 is used instead which states that it is fully compatible without conversion script needed.
[/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:20:3-20:98:writable] Warning: Connector switchOffSignal1 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/Electrical/Controls/Basics/SwitchOff/SwitchOffLogic.mo:21:3-21:125:writable] Warning: Connector switchOffSignal2 is not balanced: The number of potential variables (1) is not equal to the number

Excessive output truncated after 545424 bytes.